In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA06 - Donations
   
   BUSINESS QUESTION: 
   During the assessment period, did the Unit, either directly or through the use of 
   third parties, make any domestic/international charitable donations?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. FINANCE DATA ONLY: Unions the 6 Finance files. Coupa data is entirely excluded 
      based on the business logic.
   3. CATEGORY CODE FILTER: Strictly filters the Finance data for Category Codes 
      '292' and '427' (Charitable Donations).
   4. MAPPING & OUTPUT: Maps flagged transactions to AU_IDs using the CC Bootstrap 
      (with safe Integer casting on cost_center_id). LEFT JOINS this mapped data back 
      to the Master AU anchor. Outputs 'Yes' if a match is found, 'No' otherwise.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Finance_Raw AS (
    -- 2. Union all 6 Finance files into one master dataset
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Donation_Transactions AS (
    -- 3. Clean the Finance columns and filter STRICTLY for donation category codes
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' AND LENGTH(TRIM(CostCenter)) < 4 THEN LPAD(TRIM(CostCenter), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRIM(CatCode) AS CatCode
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN ('292', '427')
      AND CostCenter IS NOT NULL
),

CC_Mapping AS (
    -- Standardize the CC Mapping table using the correct cost_center_id column
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' AND LENGTH(TRIM(cost_center_id)) < 4 THEN LPAD(TRIM(cost_center_id), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_AUs AS (
    -- Map the donation transactions to their respective AU_IDs
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Donation_Transactions d
    INNER JOIN CC_Mapping m
        ON d.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
)

-- 4. Final Output: Strictly structured to the requested 6 columns anchored to Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA06' AS QuestionID,               
    CASE 
        WHEN COALESCE(f.Flagged_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA06 Drill-Down
   
   PURPOSE: Shows EVERY donation transaction (CatCode 292 or 427) from Finance, 
   regardless of whether the Cost Center maps to an AU, or whether that AU exists 
   in the Master List. Useful for tracking unmapped charitable donations.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Finance_Parsed AS (
    -- Extract the Raw String and parsed elements, filtered strictly for Donations
    SELECT 
        CostCenter AS Raw_String,
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' AND LENGTH(TRIM(CostCenter)) < 4 THEN LPAD(TRIM(CostCenter), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRIM(CatCode) AS CatCode,
        'Finance' AS Source_System
    FROM hive_metastore.ra_adido_2025.f25_finance_data_1 WHERE TRIM(CatCode) IN ('292', '427')
    UNION ALL SELECT CostCenter, CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' AND LENGTH(TRIM(CostCenter)) < 4 THEN LPAD(TRIM(CostCenter), 4, '0') ELSE UPPER(TRIM(CostCenter)) END, TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_2 WHERE TRIM(CatCode) IN ('292', '427')
    UNION ALL SELECT CostCenter, CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' AND LENGTH(TRIM(CostCenter)) < 4 THEN LPAD(TRIM(CostCenter), 4, '0') ELSE UPPER(TRIM(CostCenter)) END, TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_3 WHERE TRIM(CatCode) IN ('292', '427')
    UNION ALL SELECT CostCenter, CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' AND LENGTH(TRIM(CostCenter)) < 4 THEN LPAD(TRIM(CostCenter), 4, '0') ELSE UPPER(TRIM(CostCenter)) END, TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_4 WHERE TRIM(CatCode) IN ('292', '427')
    UNION ALL SELECT CostCenter, CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' AND LENGTH(TRIM(CostCenter)) < 4 THEN LPAD(TRIM(CostCenter), 4, '0') ELSE UPPER(TRIM(CostCenter)) END, TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_5 WHERE TRIM(CatCode) IN ('292', '427')
    UNION ALL SELECT CostCenter, CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' AND LENGTH(TRIM(CostCenter)) < 4 THEN LPAD(TRIM(CostCenter), 4, '0') ELSE UPPER(TRIM(CostCenter)) END, TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_6 WHERE TRIM(CatCode) IN ('292', '427')
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' AND LENGTH(TRIM(cost_center_id)) < 4 THEN LPAD(TRIM(cost_center_id), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

SELECT 
    COALESCE(m.AU_ID, 'UNMAPPED_CC') AS BusinessID,
    COALESCE(mast.In_ABAC_AU_List, 'No') AS In_ABAC_AU_List,
    f.Cleaned_CC AS Flagged_Cost_Center,
    f.CatCode AS Donation_Category_Code,
    f.Source_System,
    f.Raw_String AS Original_Data_Value
FROM Finance_Parsed f
LEFT JOIN CC_Mapping m
    ON f.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast
    ON m.AU_ID = mast.BusinessID
WHERE f.Cleaned_CC IS NOT NULL
ORDER BY BusinessID, f.Cleaned_CC;